# 🌿 Plant Model Training & Feature Extraction
This notebook shows how features are built from raw data, and then how the classification pipeline operates.

## 1. Building the Feature Cache
The script below iterates through all raw recordings in the SQLite database. For any recording not already in 



eature_cache.csv, it reads the raw scope and spectrum CSV files, computes statistical features (like kurtosis, skewness, dominant frequency, zero-crossing rate), and appends them to the cache. This ensures the ML models don't have to re-read gigabytes of raw data on every run.

In [5]:
import pandas as pd
import numpy as np
import sqlite3
import os
from scipy.stats import skew, kurtosis
import time
from pathlib import Path

# Connect to database to find all recordings
conn = sqlite3.connect('../metadata.db')
recordings = pd.read_sql_query('SELECT recording_id, plant, stress_type, event_type FROM recordings', conn)
conn.close()

CACHE_PATH = '../data_collection/feature_cache.csv'
DATA_DIR = Path('../data')

# Load existing cache or create empty DataFrame
if os.path.exists(CACHE_PATH):
    cache_df = pd.read_csv(CACHE_PATH)
    cached_ids = set(cache_df['recording_id'].values)
else:
    cache_df = pd.DataFrame()
    cached_ids = set()

print(f'Total recordings in DB: {len(recordings)}')
print(f'Already cached: {len(cached_ids)}')

# Example feature extraction function (simplified)
def extract_basic_features(rec_id):
    scope_path = DATA_DIR / f'{rec_id}-measurement-scope.csv'
    if not scope_path.exists():
        return None
    
    try:
        # We only read the first few rows for demonstration in this notebook
        # In reality, this reads the entire waveform to calculate features
        df = pd.read_csv(scope_path, on_bad_lines='skip', nrows=50000)
        voltages = df.iloc[:, 1].values
        return {
            'recording_id': rec_id,
            'wf_kurtosis': float(kurtosis(voltages, nan_policy='omit')),
            'wf_skewness': float(skew(voltages, nan_policy='omit')),
            'wf_max_amp': float(np.max(np.abs(voltages)))
        }
    except:
        return None

# Find missing recordings and extract (Showing a simulated run)
missing = [rid for rid in recordings['recording_id'] if rid not in cached_ids]
print(f'Found {len(missing)} new recordings to extract.')

# Uncomment the following to actually run extraction in the notebook:
# new_features = []
# for rec_id in missing[:5]:  # Just extracting 5 for demonstration
#     feats = extract_basic_features(rec_id)
#     if feats:
#         new_features.append(feats)
# 
# if new_features:
#     new_df = pd.DataFrame(new_features)
#     cache_df = pd.concat([cache_df, new_df], ignore_index=True)
#     cache_df.to_csv(CACHE_PATH, index=False)
#     print('Updated cache saved!')


Total recordings in DB: 23112
Already cached: 23112
Found 0 new recordings to extract.


C:\Users\mariu\AppData\Local\Temp\ipykernel_8200\3680496993.py:19: DtypeWarning: Columns (0: species, 1: pred_plant, 2: pred_stress, 3: stress_type_details, 4: plant_id) have mixed types. Specify dtype option on import or set low_memory=False.
  cache_df = pd.read_csv(CACHE_PATH)


## 2. Load the Fully Built Cache and Clean Labels

In [7]:
import pandas as pd
import plotly.express as px
import sqlite3

# Load feature cache
df = pd.read_csv('../data_collection/feature_cache.csv')

# Load manual labels from DB
conn = sqlite3.connect('../metadata.db')
db_labels = pd.read_sql_query('SELECT recording_id, event_type, plant_id FROM recordings', conn)
conn.close()

if 'event_type' in df.columns:
    df = df.drop(columns=['event_type'])
df = pd.merge(df, db_labels, on='recording_id', how='left')

# Filter for genuine cavitation
df_clean = df[(df['event_type'] == 'Cavitation')].copy()
valid_plants = ['Sunflower', 'Tomato', 'Mini Pepper']
df_plants = df_clean[df_clean['plant'].isin(valid_plants)].copy()

print(f'Total Cavitation Events: {len(df_plants)}')
display(df_plants['plant'].value_counts())

Total Cavitation Events: 286


C:\Users\mariu\AppData\Local\Temp\ipykernel_8200\2776020773.py:6: DtypeWarning: Columns (0: species, 1: pred_plant, 2: pred_stress, 3: stress_type_details, 4: plant_id) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data_collection/feature_cache.csv')


plant
Tomato         128
Sunflower      112
Mini Pepper     46
Name: count, dtype: int64

## 2.5 Data Distribution by Physical Plant (plant_id)
Let's visualize exactly how many cavitation events are recorded for each physical plant pot. If the data is highly skewed (e.g. one pot has 90 events and another has 2), StratifiedGroupKFold can struggle to create balanced folds.

In [20]:
import plotly.express as px
import sqlite3
import pandas as pd

# Fetch plant_id directly from the database to guarantee it is available
conn = sqlite3.connect('../metadata.db')
db_plants = pd.read_sql_query('SELECT recording_id, plant_id FROM recordings', conn)
conn.close()

# Merge it into our working dataset
if 'plant_id' in df_plants.columns:
    df_plants = df_plants.drop(columns=['plant_id'])
df_plants = pd.merge(df_plants, db_plants, on='recording_id', how='left')

# We ensure plant_id is treated as a string/category so it plots clearly
df_plants['plant_id_str'] = df_plants['plant_id'].astype(str).str.replace('.0', '', regex=False)
dist_df = df_plants.groupby(['plant', 'plant_id_str']).size().reset_index(name='Cavitation Events')

fig = px.bar(
    dist_df, 
    x='plant_id_str', 
    y='Cavitation Events', 
    color='plant',
    title='Number of Cavitation Events per Plant ID',
    labels={'plant_id_str': 'Plant ID'},
    barmode='group'
)
fig.update_layout(xaxis={'categoryorder':'total descending', 'type': 'category'})
fig.show()

# Show text summary
display(dist_df.sort_values(['plant', 'Cavitation Events'], ascending=[True, False]))


,plant,plant_id_str,Cavitation Events
2,Mini Pepper,12,34
1,Mini Pepper,11,7
0,Mini Pepper,10,5
4,Sunflower,13,69
3,Sunflower,1,30
6,Sunflower,9,10
5,Sunflower,2,3
7,Tomato,20,85
11,Tomato,5,18
9,Tomato,3,11


## 3. Train/Test Split (Two Scenarios for Thesis)
In academic research, evaluating a model under different constraints yields valuable insights. We will evaluate two distinct scenarios:

1. **Scenario A (Intra-Plant / Calibrated):** The model is allowed to train on a subset of recordings from a plant, and is tested on remaining recordings from that *same* plant. This simulates a real-world system that is calibrated upon installation.
2. **Scenario B (Inter-Plant / Zero-Shot):** The model is trained on a set of plants and tested on completely unseen physical plants. This tests ultimate generalization but is heavily affected by sensor-to-stem coupling differences.

By reporting both, you prove a critical acoustic limitation: sensor coupling introduces massive domain shift, meaning commercial systems likely require per-plant calibration!

In [22]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
import numpy as np

feature_cols = [c for c in df_plants.columns if c.startswith('wf_') or c.startswith('spec_')]
all_features = feature_cols + ['duration', 'counts', 'dominant_frequency']
df_plants = df_plants.dropna(subset=all_features)

X = df_plants[all_features].values
le = LabelEncoder()
y = le.fit_transform(df_plants['plant'])

experiment_ids = df_plants['experiment_id'].values if 'experiment_id' in df_plants.columns else None
plant_ids_raw = df_plants['plant_id'].fillna(df_plants['experiment_id']).values if 'plant_id' in df_plants.columns else experiment_ids
plant_ids = np.array([str(p).replace('.0', '') for p in plant_ids_raw])

# SCENARIO A: Intra-Plant (Random Split)
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
train_idx_A, test_idx_A = next(skf.split(X, y))
X_train_A, X_test_A, y_train_A, y_test_A = X[train_idx_A], X[test_idx_A], y[train_idx_A], y[test_idx_A]

# SCENARIO B: Inter-Plant (Leave-One-Out / Zero Shot)
train_plants = ['20', '13', '42', '1', '5', '3', '2', '11'] # Swapped 12 out for 42
train_idx_B = np.where(np.isin(plant_ids, train_plants))[0]
test_idx_B = np.where(~np.isin(plant_ids, train_plants))[0]
X_train_B, X_test_B, y_train_B, y_test_B = X[train_idx_B], X[test_idx_B], y[train_idx_B], y[test_idx_B]

print(f'Scenario A (Calibrated) -> Train: {len(X_train_A)}, Test: {len(X_test_A)}')
print(f'Scenario B (Zero-Shot)  -> Train: {len(X_train_B)}, Test: {len(X_test_B)}')



Unique Physical Plants (Groups): 15
Train size: 264, Test size: 22
Training on plants: ['1' '11' '12' '13' '20' '3' '5' '9']
Testing on plants:  ['10' '2' '21' '4' '6' '7' '8']


## Exploratory Data Analysis (EDA)
Before training, it is crucial to understand how the features separate the different plant species. The cells below generate interactive graphics to explore feature distributions, correlations, and the UMAP 3D projection.

In [23]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Feature Distributions (Violin Plots)
top_features = ['spec_rel_energy_0.4_0.5', 'spec_flatness', 'spec_rolloff', 'duration', 'dominant_frequency', 'wf_zcr']

fig = make_subplots(rows=1, cols=6, subplot_titles=top_features)

for i, feature in enumerate(top_features):
    for plant in df_plants['plant'].unique():
        plant_data = df_plants[df_plants['plant'] == plant][feature]
        fig.add_trace(go.Violin(y=plant_data, name=plant, box_visible=True, meanline_visible=True), row=1, col=i+1)

fig.update_layout(height=500, title_text='Feature Distributions by Plant Species', showlegend=False)
fig.show()

In [ ]:
# Feature Correlation Heatmap
# Highly correlated features can confuse models and bloat the feature space
corr_matrix = df_plants[all_features].corr()

fig_corr = px.imshow(corr_matrix, 
                     title='Feature Correlation Heatmap',
                     color_continuous_scale='RdBu_r', 
                     zmin=-1, zmax=1)
fig_corr.update_layout(height=800, width=800)
fig_corr.show()

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import RobustScaler
import umap
import numpy as np

def evaluate_scenario(name, X_tr, X_te, y_tr, y_te):
    scaler = RobustScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)
    
    # We use fewer UMAP components to prevent overfitting the small dataset
    reducer = umap.UMAP(n_components=2, random_state=42, n_jobs=1)
    X_tr_u = np.hstack((X_tr_s, reducer.fit_transform(X_tr_s)))
    X_te_u = np.hstack((X_te_s, reducer.transform(X_te_s)))
    
    print(f'\n========== {name} ==========')
    xgb = XGBClassifier(n_estimators=100, max_depth=3, random_state=42)
    xgb.fit(X_tr_u, y_tr)
    y_pred_xgb = xgb.predict(X_te_u)
    print(classification_report(y_te, y_pred_xgb, target_names=le.classes_))

# Standard Evaluation
evaluate_scenario('SCENARIO A: Intra-Plant (Calibrated)', X_train_A, X_test_A, y_train_A, y_test_A)
evaluate_scenario('SCENARIO B: Inter-Plant (Zero-Shot - ALL FEATURES)', X_train_B, X_test_B, y_train_B, y_test_B)

# SCENARIO C: Feature Engineering (Filtering out Pot Noise)
# We drop all features related to absolute volume/amplitude (which vary wildly based on sensor tightness)
# and ONLY keep relative spectral features (frequencies, shape, entropy)
invariant_prefixes = ('spec_rel', 'spec_centroid', 'spec_flatness', 'spec_entropy', 'wf_zcr', 'wf_entropy', 'wf_shape_factor')
invariant_indices = [i for i, col in enumerate(all_features) if any(col.startswith(p) for p in invariant_prefixes)]

X_train_C = X_train_B[:, invariant_indices]
X_test_C = X_test_B[:, invariant_indices]

evaluate_scenario('SCENARIO C: Inter-Plant (Zero-Shot - INVARIANT FEATURES ONLY)', X_train_C, X_test_C, y_train_B, y_test_B)


NameError: name 'X_train_umap' is not defined

## 4. Feature Scaling & UMAP
We scale the features robustly and inject UMAP dimensionality reduction.

In [24]:
from sklearn.preprocessing import RobustScaler
import umap

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

reducer = umap.UMAP(n_components=3, random_state=42)
X_train_umap = reducer.fit_transform(X_train_scaled)
X_test_umap = reducer.transform(X_test_scaled)

X_train_final = np.hstack((X_train_scaled, X_train_umap))
X_test_final = np.hstack((X_test_scaled, X_test_umap))

print(f'Final Features: {X_train_final.shape[1]}')

C:\Users\mariu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Final Features: 65


## 5. Model Training & Evaluation
Train XGBoost and evaluate on the strictly held-out physical plant.

In [25]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

model = XGBClassifier(n_estimators=100, max_depth=3, random_state=42, eval_metric='mlogloss')
model.fit(X_train_final, y_train)

y_pred = model.predict(X_test_final)
print(classification_report(y_test, y_pred, target_names=['Mini Pepper', 'Sunflower', 'Tomato']))


              precision    recall  f1-score   support

 Mini Pepper       0.50      0.20      0.29         5
   Sunflower       0.00      0.00      0.00         3
      Tomato       0.65      0.79      0.71        14

    accuracy                           0.55        22
   macro avg       0.38      0.33      0.33        22
weighted avg       0.53      0.55      0.52        22

